In [2]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

In [3]:
from models import Ride, ride_deserializer

In [4]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer
)

In [5]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [6]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...


OperationalError: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
